In [1]:
%cd ..

/Users/po3na4skld/PycharmProjects/Personal/SynSteg


In [10]:
import json

from glob import glob
import numpy as np

from models.report import ReportModel
from steganography.core import decode_message

In [3]:
JSONS_PATH = 'json_reports/*.json'
jsons = glob(JSONS_PATH)

In [8]:
REPORTS = [read_report(report_path) for report_path in jsons] 

In [84]:
TOKEN_SMALL_REPORT = read_token_report('tokens_64.txt')
TOKEN_MEDIUM_REPORT = read_token_report('tokens_128.txt')
TOKEN_BIG_REPORT = read_token_report('tokens_256.txt')

CONTAINER_COMPLETION_PRICE = 0.0020
CONTAINER_PROMPT_PRICE = 0.0010

SECRET_KEY_COMPLETION_PRICE = 0.03
SECRET_KEY_PROMPT_PRICE = 0.01

In [93]:
def read_report(path_to_report: str) -> ReportModel:
    with open(path_to_report) as f:
        report = ReportModel(**json.load(f))
        if not report.decoded_message:
            report.decoded_message = decode_message(report.encoded_message, report.secret_key, clean_output=False)
        return report

def get_throughput(report: ReportModel) -> float:
    """Function that calculates throughtput: volume of secret message / volume of the container"""
    return len(report.message) / len(report.container.split()) * 2


def get_mean_and_std(sequence: list[float], title: str) -> None:
    print(title)
    print('Mean:', np.mean(sequence), 'STD:', np.std(sequence))
    
    
def read_token_report(path_to_file: str) -> list[list[str]]:
    with open(path_to_file) as f:
        report = []
        for line in f.readlines():
            if not line.startswith(('CONTAINER', 'SECRET KEY')):
                container_idx = line.find('CONTAINER')
                secret_key_idx = line.find('SECRET KEY')
                if container_idx > 0 or secret_key_idx > 0:
                    report.append(line[max(container_idx, secret_key_idx):].strip().split())
            else:
                report.append(line.strip().split())
    return report


def get_metrics(
    reports: list[ReportModel], message_length: int, token_report: list[list] = None):
    
    filtered_reports = [report for report in reports if len(report.message) == message_length]
    
    small_throughput = [get_throughput(report) for report in filtered_reports]
    get_mean_and_std(small_throughput, f'Throughput {message_length} bits, bit/word')
    print('--' * 10)
    speed = [report.encoding_time_report['container_generation'] for report in filtered_reports]
    get_mean_and_std(speed, 'Container generation speed, seconds')
    print('--' * 10)
    speed = [report.encoding_time_report['secret_key_generation'] for report in filtered_reports]
    get_mean_and_std(speed, 'Secret key generation speed, seconds')
    print('--' * 10)
    speed = [report.decoding_time for report in filtered_reports]
    get_mean_and_std(speed, 'Decoding/Encoding time speed, seconds')
    print('--' * 10)
    secret_key_length = [len(report.secret_key) for report in filtered_reports]
    get_mean_and_std(secret_key_length, 'Secret key and Container length, words')
    print('--' * 10)
    if token_report:
        completion_tokens = [int(i[1]) for i in token_report if i[0] == 'CONTAINER']
        prompt_tokens = [int(i[2]) for i in token_report if i[0] == 'CONTAINER']

        get_mean_and_std(completion_tokens, 'Completion tokens number for Container generation')
        print('--' * 10)


        get_mean_and_std(
            [i / 1000 * CONTAINER_COMPLETION_PRICE for i in completion_tokens], 
            'Completion tokens USD price for Container generation'
        )
        print('--' * 10)

        get_mean_and_std(prompt_tokens, 'Prompt tokens number for Container generation')
        print('--' * 10)

        get_mean_and_std(
            [i / 1000 * CONTAINER_PROMPT_PRICE for i in completion_tokens], 
            'Prompt tokens USD price for Container generation'
        )
        print('--' * 10)

        secret_key_tokens = [int(i[2]) for i in token_report if i[0] == 'SECRET']
        secret_key_prompt_tokens = [int(i[3]) for i in token_report if i[0] == 'SECRET']

        get_mean_and_std(secret_key_tokens, 'Completion tokens number for Secret Key generation')
        print('--' * 10)

        get_mean_and_std(
            [i / 1000 * SECRET_KEY_COMPLETION_PRICE for i in secret_key_tokens], 
            'Completion tokens USD price for Secret Key generation'
        )
        print('--' * 10)


        get_mean_and_std(secret_key_prompt_tokens, 'Prompt tokens number for Secret Key generation')
        print('--' * 10)

        get_mean_and_std(
            [i / 1000 * SECRET_KEY_PROMPT_PRICE for i in secret_key_prompt_tokens], 
            'Prompt tokens USD price for Secret Key generation'
        )
        print('--' * 10)
    
    secret_key_average_word_size = []
    for report in filtered_reports:
        for secret_key_token in report.secret_key:
            for v in secret_key_token.values():
                for k in v:
                    secret_key_average_word_size.append(len(k))
                    break
                break

    get_mean_and_std(secret_key_average_word_size, 'Secret key token bits size')
    print('--' * 10)
    print('End of metrics')

## Metrics for 64 bit message length

In [94]:
get_metrics(REPORTS, 64, TOKEN_SMALL_REPORT)

Throughput 64 bits, bit/word
Mean: 1.2141745522652314 STD: 0.0892190224765825
--------------------
Container generation speed, seconds
Mean: 3.9835 STD: 1.1958882681923093
--------------------
Secret key generation speed, seconds
Mean: 113.06770000000002 STD: 25.663823481897627
--------------------
Decoding/Encoding time speed, seconds
Mean: 0.00035492420196533205 STD: 5.827536549530568e-05
--------------------
Secret key and Container length, words
Mean: 106.48 STD: 15.326760910251062
--------------------
Completion tokens number for Container generation
Mean: 125.14851485148515 STD: 18.151390934136117
--------------------
Completion tokens USD price for Container generation
Mean: 0.00025029702970297035 STD: 3.630278186827223e-05
--------------------
Prompt tokens number for Container generation
Mean: 85.10891089108911 STD: 1.8179576730415077
--------------------
Prompt tokens USD price for Container generation
Mean: 0.00012514851485148518 STD: 1.8151390934136116e-05
-----------------

## Metrics for 128 bit message length

In [95]:
get_metrics(REPORTS, 128, None)

Throughput 128 bits, bit/word
Mean: 1.0507688856406934 STD: 0.12172737162440708
--------------------
Container generation speed, seconds
Mean: 21.217 STD: 5.37074678233856
--------------------
Secret key generation speed, seconds
Mean: 113.9826 STD: 203.90828488622034
--------------------
Decoding/Encoding time speed, seconds
Mean: 0.0006866478919982911 STD: 0.0002493019731027763
--------------------
Secret key and Container length, words
Mean: 247.1 STD: 30.253429557655114
--------------------
Secret key token bits size
Mean: 1.4910157830837718 STD: 0.5041914766005801
--------------------
End of metrics


In [100]:
# Tokens metrics for 128 bit message length
(204.261386,
 24.1004375,
 204.261386 / 1000 * CONTAINER_COMPLETION_PRICE,
 24.1004375 / 1000 * CONTAINER_COMPLETION_PRICE,
 85.10891089108911,
 1.656035130316372,
 85.10891089108911 / 1000 * CONTAINER_PROMPT_PRICE,
 1.656035130316372 / 1000 * CONTAINER_PROMPT_PRICE,
 1090.15456,
 569.251556,
 1090.15456 / 1000 * SECRET_KEY_COMPLETION_PRICE,
 569.251556 / 1000 * SECRET_KEY_COMPLETION_PRICE,
504.038193,
125.141189,
504.038193 / 1000 * SECRET_KEY_PROMPT_PRICE,
125.141189 / 1000 * SECRET_KEY_PROMPT_PRICE
)

(204.261386,
 24.1004375,
 0.000408522772,
 4.8200875000000006e-05,
 85.10891089108911,
 1.656035130316372,
 8.510891089108912e-05,
 1.656035130316372e-06,
 1090.15456,
 569.251556,
 0.0327046368,
 0.017077546680000003,
 504.038193,
 125.141189,
 0.00504038193,
 0.00125141189)

## Metrics for 256 bit message length

In [96]:
get_metrics(REPORTS, 256, TOKEN_BIG_REPORT)

Throughput 256 bits, bit/word
Mean: 1.2259096675328882 STD: 0.08113064588199323
--------------------
Container generation speed, seconds
Mean: 30.8911 STD: 71.2837065099031
--------------------
Secret key generation speed, seconds
Mean: 176.5882 STD: 81.05359265794452
--------------------
Decoding/Encoding time speed, seconds
Mean: 0.0013623762130737305 STD: 0.0003013252032096714
--------------------
Secret key and Container length, words
Mean: 419.5 STD: 28.12134420684758
--------------------
Completion tokens number for Container generation
Mean: 484.7402597402597 STD: 30.049484122633977
--------------------
Completion tokens USD price for Container generation
Mean: 0.0009694805194805196 STD: 6.009896824526794e-05
--------------------
Prompt tokens number for Container generation
Mean: 85.1038961038961 STD: 1.656035130316372
--------------------
Prompt tokens USD price for Container generation
Mean: 0.0004847402597402598 STD: 3.004948412263397e-05
--------------------
Completion toke